# Hybrid quantum–classical OCT classification — notebook 01: backbone, baseline VQC head and classical controls

Companion code for the paper. This notebook reproduces **Section A** of the results (see `README.md` for the full map):
the patient-level split, the fine-tuned ResNet-18d backbone, the 4 × 5-qubit, 2-layer VQC head against its classical controls
(5-fold patient-grouped CV × 3 seeds, locked internal test, official Kermany test), data efficiency, image-noise robustness and
simulated quantum noise (finite shots, depolarising channels).

**Design.**

| Element | Choice |
|---|---|
| Split | Patients shared between the Kermany train and test folders are removed from every training pool. The remaining train-folder patients are partitioned (patient level, stratified) into backbone / backbone-val / head / **locked test** pools. The locked-test file list is hashed (SHA-256 of the sorted file names). |
| Backbone | `resnet18d.ra2_in1k`, full fine-tune on the backbone pool (AMP, class-balanced sampling, cosine LR, early stopping on backbone-val), then frozen. |
| Heads (same frozen 512-d features, same encoder Linear(512→20), same recipe) | `hybrid_vqc` (4 parallel 5-qubit circuits, RY embedding of π·tanh(z), 2 StronglyEntanglingLayers, ⟨Y⟩ readout); `classical_matched` (parameter-matched 120-parameter classical block); `hybrid_vqc_frozen` (random, untrained circuit); `bottleneck_only`; logistic regression. |
| Model selection | 5-fold StratifiedGroupKFold by patient × 3 seeds on the head pool only. |
| Statistics | Nadeau–Bengio corrected resampled *t*-test on CV, McNemar (image-level and patient-clustered) and patient-cluster bootstrap on the test sets, Holm correction. |
| Quantum noise | Finite-shot sampling and single-qubit depolarising channels (exact density-matrix simulation). |
| Image noise | Additive Gaussian and multiplicative speckle noise in pixel space; the backbone never sees noise during training. |

The quantum circuit is simulated exactly in PyTorch (batched, GPU) and **checked against PennyLane** (`default.qubit` / `default.mixed`)
for outputs and gradients before anything is trained with it.

**Outputs** go to the results folder (`RESULTS_DIR`; on Kaggle `/kaggle/working/results`). Notebook 02 reads `features.npz` and the
backbone checkpoint from the same folder. Environment variable `FAST_DEV_RUN=1` runs the whole notebook on a tiny subset in minutes
(pipeline check only; the numbers are meaningless).

In [ ]:
# Cell 0: Google Colab setup (does nothing on Kaggle or a local machine)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/oct_qml/results'    # persists across disconnects
    os.makedirs(os.environ['RESULTS_DIR'], exist_ok=True)
    for k in ['KAGGLE_API_TOKEN']:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            print(f'Colab secret {k} not set')
    if os.environ.get('KAGGLE_API_TOKEN'):
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/access_token', 'w') as f:
            f.write(os.environ['KAGGLE_API_TOKEN'])
        os.chmod('/root/.kaggle/access_token', 0o600)
    get_ipython().system('pip install -q -U kaggle kagglehub')
    try:
        import kagglehub
        path = kagglehub.dataset_download('paultimothymooney/kermany2018')
    except Exception as e:
        print('kagglehub failed (', e, ') -> using the kaggle CLI')
        get_ipython().system('kaggle datasets download -d paultimothymooney/kermany2018 -p /content/data')
        get_ipython().system('unzip -q -o /content/data/kermany2018.zip -d /content/data')
        path = '/content/data'
    roots = [d for d, sub, _ in os.walk(path) if {'train', 'test'} <= set(sub) and '__MACOSX' not in d
             and all(os.path.isdir(os.path.join(d, 'train', c)) for c in ['NORMAL', 'CNV', 'DME', 'DRUSEN'])]
    assert roots, f'Kermany train/test folders not found under {path}'
    os.environ['OCT_ROOT'] = sorted(roots, key=len)[0]
    print('dataset:', repr(os.environ['OCT_ROOT']), '| results ->', os.environ['RESULTS_DIR'])
else:
    print('not in Colab -> Kaggle input, or OCT_ROOT / RESULTS_DIR environment variables, are used')

In [ ]:
# Cell 1: dependencies (PennyLane is used only to verify the simulator and draw the circuit)
!pip install -q pennylane timm

In [ ]:
# Cell 2: imports, configuration, reproducibility
import os, glob, json, math, time, copy, random, hashlib, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import timm
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore', category=UserWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
FAST_DEV_RUN = bool(int(os.environ.get('FAST_DEV_RUN', '0')))   # tiny pipeline smoke test only
UNLOCK_TEST = True   # the reported run: the test sets are opened once, by Cell 14

GLOBAL_SEED = 42
CLASSES = ['NORMAL', 'CNV', 'DME', 'DRUSEN']
K = len(CLASSES)

CFG = dict(
    backbone='resnet18d.ra2_in1k', pretrained=True, img_size=224,
    # patient-level partition of the official *train* folder
    locked_test_frac=0.15, backbone_frac=0.50,          # remainder -> head pool (CV)
    bb_val_frac=0.10,                                    # of backbone pool, for early stopping
    # backbone fine-tuning
    bb_epochs=12, bb_patience=3, bb_bs=64, bb_lr_head=1e-3, bb_lr_body=2e-4, bb_wd=0.05,
    bb_samples_per_epoch=40000, label_smoothing=0.1, reuse_backbone=True,
    # quantum layer (HQNN-Parallel)
    q_circuits=4, q_qubits=5, q_layers=2, q_dropout=0.1,
    # heads
    n_folds=5, head_seeds=[0, 1, 2], head_epochs=40, head_bs=256, head_lr=3e-3,
    head_wd=1e-4, head_patience=6, head_in_dropout=0.2, head_min_steps=0,
    # experiments
    data_eff_per_class=[25, 100, 400, 1600], data_eff_min_steps=800,
    img_noise_levels=[0.0, 0.05, 0.10, 0.20, 0.30], noise_eval_per_class=500,
    shot_levels=[4096, 1024, 256, 64, 16], shot_repeats=5,
    depol_levels=[0.0, 1e-3, 5e-3, 1e-2, 2e-2, 5e-2],
    n_boot=2000, num_workers=4,
)
if FAST_DEV_RUN:
    CFG.update(bb_epochs=1, bb_samples_per_epoch=256, bb_bs=16, head_epochs=3, head_min_steps=300, head_seeds=[0],
               n_folds=3, data_eff_per_class=[5, 10], data_eff_min_steps=20,
               noise_eval_per_class=8, img_noise_levels=[0.0, 0.2], shot_levels=[64],
               shot_repeats=1, depol_levels=[0.0, 1e-2], n_boot=50, num_workers=0,
               reuse_backbone=False)

OUT = os.environ.get('RESULTS_DIR') or ('/kaggle/working/results' if os.path.isdir('/kaggle/working') else os.path.abspath('results'))
os.makedirs(OUT, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # speed; head experiments are seeded separately
    torch.backends.cudnn.benchmark = True
seed_everything(GLOBAL_SEED)

print('device:', DEVICE, '| AMP:', AMP, '| FAST_DEV_RUN:', FAST_DEV_RUN, '| UNLOCK_TEST:', UNLOCK_TEST)
if DEVICE.type != 'cuda' and not FAST_DEV_RUN:
    print('WARNING: no GPU. Enable a GPU accelerator (T4/P100) -- backbone fine-tuning on CPU is not viable.')
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)

In [ ]:
# Cell 3: index images, remove patients shared with the official test set, patient-level partition
def find_root():
    # Search /kaggle/input (any depth) for a folder holding train/ and test/ with the class subfolders.
    env = os.environ.get('OCT_ROOT')
    if env and os.path.isdir(os.path.join(env, 'train')):
        return env
    base = '/kaggle/input'
    hits = []
    for dirpath, dirnames, _ in os.walk(base):
        if dirpath[len(base):].count(os.sep) > 6:
            dirnames[:] = []; continue
        if 'train' in dirnames and 'test' in dirnames and \
                all(os.path.isdir(os.path.join(dirpath, 'train', c)) for c in CLASSES):
            hits.append(dirpath); dirnames[:] = []
    if hits:
        hits.sort(key=lambda p: ('OCT' not in p, len(p)))   # prefer the OCT2017 folder over duplicates
        if len(hits) > 1:
            print('several candidate roots found, using the first:', hits)
        return hits[0]
    print('Nothing usable under /kaggle/input. What is attached:')
    if os.path.isdir(base):
        for dirpath, dirnames, files in os.walk(base):
            depth = dirpath[len(base):].count(os.sep)
            if depth > 3:
                dirnames[:] = []; continue
            print('  ' * depth + os.path.basename(dirpath) + '/', f'({len(files)} files)')
    else:
        print('  /kaggle/input does not exist')
    raise FileNotFoundError(
        'Kermany OCT2017 not found. In the Kaggle editor: Add Input -> search "kermany2018" '
        '(paultimothymooney) -> Add, then re-run this cell. Or set OCT_ROOT to the folder containing train/ and test/.')
ROOT = find_root(); print('data root:', repr(ROOT))

def patient_id(path):            # Kermany filenames: CLASS-PATIENTID-N.jpeg
    return os.path.basename(path).split('-')[1]

def strat_sample(df, n_per_class, seed):
    parts = [d.sample(min(len(d), n_per_class), random_state=seed) for _, d in df.groupby('label')]
    return pd.concat(parts).sort_index().reset_index(drop=True)

def index_split(split):
    rows = []
    for lab, c in enumerate(CLASSES):
        for f in sorted(glob.glob(os.path.join(ROOT, split, c, '*.jpeg'))):
            rows.append((f, lab, patient_id(f)))
    return pd.DataFrame(rows, columns=['path', 'label', 'patient'])

train_all = index_split('train')
official = index_split('test')

if FAST_DEV_RUN:
    train_all = strat_sample(train_all, 120, GLOBAL_SEED)
    official = strat_sample(official, 20, GLOBAL_SEED)

shared = set(train_all.patient) & set(official.patient)
print(f'train folder: {len(train_all)} imgs / {train_all.patient.nunique()} patients; '
      f'official test: {len(official)} imgs / {official.patient.nunique()} patients')
print(f'patients present in BOTH train folder and official test: {len(shared)} '
      f'-> {train_all.patient.isin(shared).sum()} train-folder images removed')
train_all = train_all[~train_all.patient.isin(shared)].reset_index(drop=True)

# Patient-level partition, stratified by each patient's majority label.
# A patient can have scans in several classes; the patient (not the image) is the unit.
pat_label = train_all.groupby('patient')['label'].agg(lambda s: s.value_counts().idxmax())
rng = np.random.RandomState(GLOBAL_SEED)
assign = {}
for lab, grp in pat_label.groupby(pat_label):
    pats = np.array(sorted(grp.index)); rng.shuffle(pats)
    n = len(pats)
    a = int(round(n * CFG['locked_test_frac'])); b = a + int(round(n * CFG['backbone_frac']))
    for p in pats[:a]:  assign[p] = 'locked_test'
    for p in pats[a:b]: assign[p] = 'backbone'
    for p in pats[b:]:  assign[p] = 'head'
train_all['pool'] = train_all.patient.map(assign)

# backbone-val (early stopping only) = patient-disjoint slice of the backbone pool
bb_pats = pat_label[pat_label.index.isin(train_all.loc[train_all.pool == 'backbone', 'patient'])]
for lab, grp in bb_pats.groupby(bb_pats):
    pats = np.array(sorted(grp.index)); rng.shuffle(pats)
    for p in pats[:max(1, int(round(len(pats) * CFG['bb_val_frac'])))]:
        assign[p] = 'backbone_val'
train_all['pool'] = train_all.patient.map(assign)

POOLS = {k: train_all[train_all.pool == k].reset_index(drop=True)
         for k in ['backbone', 'backbone_val', 'head', 'locked_test']}
POOLS['official_test'] = official.reset_index(drop=True)

names = list(POOLS)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        assert not set(POOLS[a].patient) & set(POOLS[b].patient), f'patient overlap {a}/{b}'
        assert not set(POOLS[a].path) & set(POOLS[b].path), f'image overlap {a}/{b}'

summary = pd.DataFrame({k: v.label.value_counts().reindex(range(K), fill_value=0).values
                        for k, v in POOLS.items()}, index=CLASSES).T
summary['images'] = summary.sum(axis=1); summary['patients'] = [v.patient.nunique() for v in POOLS.values()]
print(summary.to_string()); print('All pools are image- and patient-disjoint.')

locked_hash_full = hashlib.sha256('\n'.join(sorted(POOLS['locked_test'].path.map(os.path.basename))).encode()).hexdigest()   # path-independent
locked_hash = locked_hash_full[:16]
print('locked-test SHA-256 (filenames only):', locked_hash_full)
print('  -> paper uses the first 16 characters:', locked_hash)
json.dump({k: sorted(v.path.map(os.path.basename)) for k, v in POOLS.items()},
          open(f'{OUT}/split_manifest.json', 'w'))
json.dump({k: sorted(v.path.map(os.path.basename)) for k, v in POOLS.items()},
          open(f'{OUT}/split_manifest.json', 'w'))
summary.to_csv(f'{OUT}/split_summary.csv')

In [ ]:
# Cell 4: transforms, pixel-space noise, dataset
_cfg = timm.data.resolve_model_data_config(timm.create_model(CFG['backbone'], pretrained=False))
MEAN, STD = float(np.mean(_cfg['mean'])), float(np.mean(_cfg['std']))  # grayscale -> one mean/std
S = CFG['img_size']

train_tf = T.Compose([
    T.RandomResizedCrop(S, scale=(0.80, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(), T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2), T.ToTensor()])
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor()])

def add_pixel_noise(x, kind, level, gen):
    # x: (1,H,W) in [0,1] BEFORE normalisation. gaussian: x + s*n ; speckle (multiplicative): x*(1 + s*n)
    if level <= 0:
        return x
    n = torch.randn(x.shape, generator=gen)
    x = x + level * n if kind == 'gaussian' else x * (1.0 + level * n)
    return x.clamp(0.0, 1.0)

class OCTDataset(Dataset):
    def __init__(self, df, train=False, noise=None):
        self.paths, self.labels = df.path.tolist(), df.label.tolist()
        self.tf = train_tf if train else eval_tf
        self.noise = noise                       # None or (kind, level)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        x = self.tf(Image.open(self.paths[i]).convert('L'))
        if self.noise is not None:
            # same noise field per image across levels -> only the amplitude changes
            g = torch.Generator().manual_seed(GLOBAL_SEED * 1_000_003 + i)
            x = add_pixel_noise(x, *self.noise, g)
        x = (x - MEAN) / STD
        return x.expand(3, -1, -1), self.labels[i]

def make_loader(df, train=False, bs=128, sampler=None, noise=None):
    return DataLoader(OCTDataset(df, train, noise), batch_size=bs, sampler=sampler,
                      shuffle=False, num_workers=CFG['num_workers'], pin_memory=DEVICE.type == 'cuda',
                      drop_last=train, persistent_workers=CFG['num_workers'] > 0)

def metrics(y, prob):
    pred = np.asarray(prob).argmax(1)
    return dict(acc=accuracy_score(y, pred), bacc=balanced_accuracy_score(y, pred),
                f1=f1_score(y, pred, average='macro'))
print('ok')

In [ ]:
# Cell 5: full fine-tuning of the backbone on the backbone pool (no test data anywhere in this cell)
def build_backbone(pretrained):
    try:
        return timm.create_model(CFG['backbone'], pretrained=pretrained, num_classes=K, drop_rate=0.2)
    except Exception as e:
        if not FAST_DEV_RUN:
            raise
        print('pretrained weights unavailable in smoke test -> random init:', type(e).__name__)
        return timm.create_model(CFG['backbone'], pretrained=False, num_classes=K, drop_rate=0.2)

@torch.no_grad()
def run_backbone(model, df, noise=None, bs=128):
    # returns pooled features (pre-logits) and softmax probabilities of the end-to-end model
    model.eval(); feats, probs = [], []
    for x, _ in make_loader(df, bs=bs, noise=noise):
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            f = model.forward_head(model.forward_features(x), pre_logits=True)
            logit = model.get_classifier()(f)
        feats.append(f.float().cpu()); probs.append(torch.softmax(logit.float(), 1).cpu())
    return torch.cat(feats).numpy(), torch.cat(probs).numpy()

CKPT = f"{OUT}/backbone_{CFG['backbone'].replace('.', '_')}.pt"
backbone = build_backbone(CFG['pretrained']).to(DEVICE).to(memory_format=torch.channels_last)
bb_hist = []

if CFG['reuse_backbone'] and os.path.exists(CKPT):
    backbone.load_state_dict(torch.load(CKPT, map_location=DEVICE)); print('loaded', CKPT)
else:
    seed_everything(GLOBAL_SEED)
    tr, va = POOLS['backbone'], POOLS['backbone_val']
    cls_w = 1.0 / tr.label.value_counts().reindex(range(K)).values
    sampler = WeightedRandomSampler(torch.tensor(cls_w[tr.label.values], dtype=torch.double),
                                    num_samples=min(len(tr), CFG['bb_samples_per_epoch']),
                                    replacement=True, generator=torch.Generator().manual_seed(GLOBAL_SEED))
    tr_loader = make_loader(tr, train=True, bs=CFG['bb_bs'], sampler=sampler)
    head_p = list(backbone.get_classifier().parameters()); head_ids = {id(p) for p in head_p}
    body_p = [p for p in backbone.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{'params': body_p, 'lr': CFG['bb_lr_body']},
                             {'params': head_p, 'lr': CFG['bb_lr_head']}], weight_decay=CFG['bb_wd'])
    total = CFG['bb_epochs'] * len(tr_loader); warm = max(1, len(tr_loader) // 2)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    scaler_amp = torch.amp.GradScaler('cuda', enabled=AMP)
    crit = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    best, bad = -1.0, 0
    for ep in range(CFG['bb_epochs']):
        backbone.train(); t0 = time.time(); tl, n = 0.0, 0
        for x, y in tr_loader:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                loss = crit(backbone(x), y)
            opt.zero_grad(set_to_none=True); scaler_amp.scale(loss).backward()
            scaler_amp.step(opt); scaler_amp.update(); sched.step()
            tl += loss.item() * len(y); n += len(y)
        _, pv = run_backbone(backbone, va)
        m = metrics(va.label.values, pv)
        bb_hist.append(dict(epoch=ep + 1, train_loss=tl / n, **{f'val_{k}': v for k, v in m.items()}))
        print(f"BB ep {ep+1:02d} loss {tl/n:.4f} | backbone-val acc {m['acc']:.4f} bacc {m['bacc']:.4f} "
              f"f1 {m['f1']:.4f} | {time.time()-t0:.0f}s")
        if m['f1'] > best:
            best, bad = m['f1'], 0; torch.save(backbone.state_dict(), CKPT)
        else:
            bad += 1
            if bad >= CFG['bb_patience']:
                print('early stop'); break
    backbone.load_state_dict(torch.load(CKPT, map_location=DEVICE))
    pd.DataFrame(bb_hist).to_csv(f'{OUT}/backbone_history.csv', index=False)
    h = pd.DataFrame(bb_hist)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.epoch, h.train_loss, 'o-'); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
    for k in ['val_acc', 'val_bacc', 'val_f1']:
        ax[1].plot(h.epoch, h[k], 'o-', label=k)
    ax[1].set_title('backbone-val (patient-disjoint)'); ax[1].legend(); ax[1].set_xlabel('epoch')
    plt.tight_layout(); plt.savefig(f'{OUT}/backbone_training.png', dpi=150); plt.show()
print('best backbone-val macro-F1 checkpoint ready')

In [ ]:
# Cell 6: frozen-backbone feature extraction (head pool, locked test, official test, noisy locked subset)
for p in backbone.parameters():
    p.requires_grad = False

FEAT = {}
for name in ['head', 'locked_test', 'official_test']:
    t0 = time.time()
    f, pr = run_backbone(backbone, POOLS[name])
    FEAT[name] = dict(X=f, bb_prob=pr, y=POOLS[name].label.values, g=POOLS[name].patient.values)
    print(f'{name:14s}: {f.shape} in {time.time()-t0:.0f}s')
FEAT_DIM = FEAT['head']['X'].shape[1]

# class-stratified subset of the locked test for the image-noise study (decoded once per noise setting)
noise_df = strat_sample(POOLS['locked_test'], CFG['noise_eval_per_class'], GLOBAL_SEED)
NOISE_FEAT = {}
for kind in ['gaussian', 'speckle']:
    for lvl in CFG['img_noise_levels']:
        if lvl == 0 and ('clean', 0.0) in NOISE_FEAT:
            NOISE_FEAT[(kind, 0.0)] = NOISE_FEAT[('clean', 0.0)]; continue
        f, pr = run_backbone(backbone, noise_df, noise=(kind, lvl))
        NOISE_FEAT[(kind, lvl)] = dict(X=f, bb_prob=pr)
        if lvl == 0:
            NOISE_FEAT[('clean', 0.0)] = NOISE_FEAT[(kind, 0.0)]
        print(f'noise {kind:8s} {lvl:.2f}: done')
NOISE_Y, NOISE_G = noise_df.label.values, noise_df.patient.values
np.savez_compressed(f'{OUT}/features.npz', **{f'{k}_{a}': v[a] for k, v in FEAT.items() for a in ['X', 'bb_prob', 'y']})

## Exact batched simulator for the quantum layer

Each of the `C` parallel circuits: `RY` angle embedding on 5 wires → `L` StronglyEntanglingLayers (PennyLane convention: `Rot(φ,θ,ω)=RZ(ω)RY(θ)RZ(φ)` on each wire, then a CNOT ring with range `(l mod (n-1))+1`) → ⟨Y⟩ on every wire.

* **Statevector path** (training, noiseless): exact, differentiable, batched over samples × circuits.
* **Density-matrix path** (evaluation): single-qubit depolarising channel with probability *p* after every gate (after a CNOT, on both its wires).
* **Finite shots**: each ⟨Y_w⟩ is replaced by its *S*-shot estimate, 2·Binomial(S, (1+⟨Y_w⟩)/2)/S − 1 (marginally exact).

The next cell checks the simulator against PennyLane before anything is trained with it.

In [ ]:
# Cell 7: torch quantum simulator
CDT = torch.complex64

def _ry(th):
    c, s = torch.cos(th / 2), torch.sin(th / 2)
    return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2).to(CDT)

def _rz(ph):
    e = torch.exp(-0.5j * ph.to(CDT)); z = torch.zeros_like(e)
    return torch.stack([torch.stack([e, z], -1), torch.stack([z, e.conj()], -1)], -2)

def _rot(w):                       # w (...,3) -> Rot(phi,theta,omega) = RZ(omega) RY(theta) RZ(phi)
    return _rz(w[..., 2]) @ _ry(w[..., 1]) @ _rz(w[..., 0])

PX = torch.tensor([[0, 1], [1, 0]], dtype=CDT)
PY = torch.tensor([[0, -1j], [1j, 0]], dtype=CDT)
PZ = torch.tensor([[1, 0], [0, -1]], dtype=CDT)

def _apply_1q(t, U, axis):
    # t: (B, C, 2, ..., 2); applies U on `axis`: out_i = sum_j U_ij t_j.  U: (2,2) | (C,2,2) | (B,C,2,2)
    B, C = t.shape[:2]
    t = t.movedim(axis, -1); shp = t.shape
    t2 = t.reshape(B, C, -1, 2)
    if U.dim() == 2:
        out = t2 @ U.transpose(0, 1)
    elif U.dim() == 3:
        out = torch.einsum('cij,bckj->bcki', U, t2)
    else:
        out = torch.einsum('bcij,bckj->bcki', U, t2)
    return out.reshape(shp).movedim(-1, axis)

def _apply_cnot(t, ac, at):
    t = t.movedim((ac, at), (-2, -1))
    t = torch.stack([t[..., 0, :], t[..., 1, :].flip(-1)], dim=-2)
    return t.movedim((-2, -1), (ac, at))

def _depol(rho, p, w, n):
    out = (1 - p) * rho
    for P in (PX.to(rho.device), PY.to(rho.device), PZ.to(rho.device)):
        out = out + (p / 3) * _apply_1q(_apply_1q(rho, P, 2 + w), P.conj(), 2 + n + w)
    return out

def qsim(angles, weights, depol=0.0, shots=None, gen=None):
    # angles (B,C,n) real; weights (C,L,n,3) real -> <Y_w> (B,C,n)
    B, C, n = angles.shape; L = weights.shape[1]; dev = angles.device
    ranges = [(l % (n - 1)) + 1 for l in range(L)] if n > 1 else [0] * L
    emb = _ry(angles)                      # (B,C,n,2,2)
    rot = _rot(weights)                    # (C,L,n,2,2)
    if depol == 0.0:
        psi = torch.zeros(B, C, 2 ** n, dtype=CDT, device=dev); psi[:, :, 0] = 1
        psi = psi.reshape(B, C, *([2] * n))
        for w in range(n):
            psi = _apply_1q(psi, emb[:, :, w], 2 + w)
        for l in range(L):
            for w in range(n):
                psi = _apply_1q(psi, rot[:, l, w], 2 + w)
            for w in range(n):
                psi = _apply_cnot(psi, 2 + w, 2 + (w + ranges[l]) % n)
        ev = []
        for w in range(n):
            a0, a1 = psi.select(2 + w, 0), psi.select(2 + w, 1)
            ev.append(2 * (a0.conj() * a1).imag.sum(dim=tuple(range(2, 1 + n))))   # <Y> = 2 Im(a0* a1)
        ev = torch.stack(ev, -1)
    else:
        rho = torch.zeros(B, C, 2 ** n, 2 ** n, dtype=CDT, device=dev); rho[:, :, 0, 0] = 1
        rho = rho.reshape(B, C, *([2] * (2 * n)))
        def U_(r, U, w):
            return _apply_1q(_apply_1q(r, U, 2 + w), U.conj(), 2 + n + w)
        for w in range(n):
            rho = _depol(U_(rho, emb[:, :, w], w), depol, w, n)
        for l in range(L):
            for w in range(n):
                rho = _depol(U_(rho, rot[:, l, w], w), depol, w, n)
            for w in range(n):
                c, t = w, (w + ranges[l]) % n
                rho = _apply_cnot(_apply_cnot(rho, 2 + c, 2 + t), 2 + n + c, 2 + n + t)
                rho = _depol(_depol(rho, depol, c, n), depol, t, n)
        ev = []
        for w in range(n):
            yr = _apply_1q(rho, PY.to(dev), 2 + w).reshape(B, C, 2 ** n, 2 ** n)
            ev.append(torch.diagonal(yr, dim1=-2, dim2=-1).sum(-1).real)
        ev = torch.stack(ev, -1)
    if shots:
        p_plus = ((1 + ev) / 2).clamp(0, 1).float()
        k = torch.binomial(torch.full_like(p_plus, float(shots)), p_plus, generator=gen)
        ev = 2 * k / shots - 1
    return ev.float()
print('simulator defined')

In [ ]:
# Cell 8: verify the simulator against PennyLane (outputs, gradients, depolarising density matrix)
import pennylane as qml
n, L = CFG['q_qubits'], CFG['q_layers']
ranges = [(l % (n - 1)) + 1 for l in range(L)]
dev_sv = qml.device('default.qubit', wires=n)
dev_dm = qml.device('default.mixed', wires=n)

@qml.qnode(dev_sv, interface='torch', diff_method='backprop')
def ref_sv(x, w):
    qml.AngleEmbedding(x, wires=range(n), rotation='Y')
    qml.StronglyEntanglingLayers(w, wires=range(n))
    return [qml.expval(qml.PauliY(i)) for i in range(n)]

@qml.qnode(dev_dm, interface='torch')
def ref_dm(x, w, p):
    for i in range(n):
        qml.RY(x[i], wires=i); qml.DepolarizingChannel(p, wires=i)
    for l in range(L):
        for i in range(n):
            qml.Rot(w[l, i, 0], w[l, i, 1], w[l, i, 2], wires=i); qml.DepolarizingChannel(p, wires=i)
        for i in range(n):
            j = (i + ranges[l]) % n
            qml.CNOT(wires=[i, j]); qml.DepolarizingChannel(p, wires=i); qml.DepolarizingChannel(p, wires=j)
    return [qml.expval(qml.PauliY(i)) for i in range(n)]

g = torch.Generator().manual_seed(0); max_err = dict(sv=0.0, grad=0.0, dm=0.0)
for trial in range(5):
    x = (torch.rand(n, generator=g) * 2 - 1) * math.pi
    w = torch.rand(L, n, 3, generator=g) * 2 * math.pi
    # forward + gradient, noiseless
    w_ref = w.clone().double().requires_grad_(True); w_ours = w.clone().requires_grad_(True)
    coef = torch.randn(n, generator=g)
    r = torch.stack(list(ref_sv(x.double(), w_ref))).float(); (r * coef).sum().backward()
    o = qsim(x.view(1, 1, n), w_ours.view(1, L, n, 3))[0, 0]; (o * coef).sum().backward()
    max_err['sv'] = max(max_err['sv'], (r.detach() - o.detach()).abs().max().item())
    max_err['grad'] = max(max_err['grad'], (w_ref.grad.float() - w_ours.grad).abs().max().item())
    # depolarising
    p = [1e-3, 1e-2, 5e-2, 0.1, 0.2][trial]
    rd = torch.stack(list(ref_dm(x.double(), w.double(), p))).float()
    od = qsim(x.view(1, 1, n), w.view(1, L, n, 3), depol=p)[0, 0]
    max_err['dm'] = max(max_err['dm'], (rd - od).abs().max().item())
print('max |torch - PennyLane| :', {k: f'{v:.2e}' for k, v in max_err.items()})
assert all(v < 1e-4 for v in max_err.values()), 'simulator does not match PennyLane'
print('Simulator matches PennyLane (statevector, gradients, depolarising density matrix).')

## Heads compared (all on identical frozen features)

All torch heads share: input dropout → `Linear(F, 20)` encoder → (middle block) → dropout → `Linear(20, 4)`; same optimiser, schedule, loss, early stopping, data.

| Head | Middle block | Trainable middle params |
|---|---|---|
| `logreg` | — (multinomial logistic regression, C tuned on fold-val) | — |
| `bottleneck_only` | `tanh(z)` | 0 |
| `classical_matched` | 4 × [`Linear(5,5)` → `tanh`] on `π·tanh(z)` | 120 |
| `hybrid_vqc_frozen` | 4 × 5-qubit circuits, **random fixed** weights | 0 (120 fixed) |
| `hybrid_vqc` | 4 × 5-qubit circuits, **trained** weights | 120 |

The question the paper can answer: does `hybrid_vqc` beat `classical_matched` (same parameter budget, same bounded output range)? If not, the honest claim is parity at equal parameter count.

In [ ]:
# Cell 9: head definitions
QC, QN, QL = CFG['q_circuits'], CFG['q_qubits'], CFG['q_layers']
QW = QC * QN   # width of the quantum layer = 20

class Head(nn.Module):
    def __init__(self, kind, in_dim, seed):
        super().__init__()
        self.kind = kind
        g = torch.Generator().manual_seed(10_000 + seed)
        self.drop_in = nn.Dropout(CFG['head_in_dropout'])
        self.enc = nn.Linear(in_dim, QW)
        self.drop_mid = nn.Dropout(CFG['q_dropout'])
        self.out = nn.Linear(QW, K)
        if kind == 'hybrid_vqc':
            self.qw = nn.Parameter(torch.rand(QC, QL, QN, 3, generator=g) * 2 * math.pi)  # PennyLane TorchLayer init
        elif kind == 'hybrid_vqc_frozen':
            self.register_buffer('qw', torch.rand(QC, QL, QN, 3, generator=g) * 2 * math.pi)
        elif kind == 'classical_matched':
            bound = 1 / math.sqrt(QN)
            self.cw = nn.Parameter((torch.rand(QC, QN, QN, generator=g) * 2 - 1) * bound)   # 4*25
            self.cb = nn.Parameter((torch.rand(QC, QN, generator=g) * 2 - 1) * bound)       # 4*5  -> 120
        elif kind != 'bottleneck_only':
            raise ValueError(kind)

    def middle(self, z, depol=0.0, shots=None, gen=None):
        B = z.shape[0]
        if self.kind == 'bottleneck_only':
            return torch.tanh(z)
        a = math.pi * torch.tanh(z)                      # bounded angles in (-pi, pi)
        if self.kind.startswith('hybrid'):
            return qsim(a.view(B, QC, QN), self.qw, depol=depol, shots=shots, gen=gen).reshape(B, QW)
        a = a.view(B, QC, QN)
        return torch.tanh(torch.einsum('cij,bcj->bci', self.cw, a) + self.cb).reshape(B, QW)

    def forward(self, x, **qkw):
        return self.out(self.drop_mid(self.middle(self.enc(self.drop_in(x)), **qkw)))

TORCH_HEADS = ['bottleneck_only', 'classical_matched', 'hybrid_vqc_frozen', 'hybrid_vqc']
ALL_HEADS = ['logreg'] + TORCH_HEADS
LABEL = {'logreg': 'Logistic regression', 'bottleneck_only': 'Bottleneck only',
         'classical_matched': 'Classical, 120 params', 'hybrid_vqc_frozen': 'VQC, frozen random',
         'hybrid_vqc': 'VQC, trained (120 params)', 'backbone_e2e': 'ResNet end-to-end'}

rows = []
for kd in TORCH_HEADS:
    h = Head(kd, FEAT_DIM, 0)
    tr = sum(p.numel() for p in h.parameters() if p.requires_grad)
    mid = sum(p.numel() for n_, p in h.named_parameters() if n_ in ('qw', 'cw', 'cb'))
    rows.append(dict(head=kd, trainable_total=tr, trainable_middle=mid))
PARAMS = pd.DataFrame(rows); print(PARAMS.to_string(index=False))
PARAMS.to_csv(f'{OUT}/param_counts.csv', index=False)

In [ ]:
# Cell 10: head training (identical recipe for every torch head) and prediction
def train_head(kind, Xtr, ytr, Xva, yva, seed, epochs=None, min_steps=None):
    seed_everything(seed)
    model = Head(kind, Xtr.shape[1], seed).to(DEVICE)
    decay = [p for n_, p in model.named_parameters() if n_ in ('enc.weight', 'out.weight')]
    other = [p for n_, p in model.named_parameters() if p.requires_grad and n_ not in ('enc.weight', 'out.weight')]
    opt = torch.optim.Adam([{'params': decay, 'weight_decay': CFG['head_wd']},
                            {'params': other, 'weight_decay': 0.0}], lr=CFG['head_lr'])
    counts = np.bincount(ytr, minlength=K).astype(float)
    cw = torch.tensor(len(ytr) / (K * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=CFG['label_smoothing'])
    N = len(ytr); bs = min(CFG['head_bs'], N); spe = math.ceil(N / bs)
    total = max((epochs or CFG['head_epochs']) * spe, min_steps or CFG['head_min_steps'])
    eval_every = max(spe, 25)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
    Xt = torch.as_tensor(Xtr, dtype=torch.float32, device=DEVICE); yt = torch.as_tensor(ytr, device=DEVICE)
    gperm = torch.Generator(device='cpu').manual_seed(seed)
    best, best_state, bad, step = -1.0, None, 0, 0
    perm = torch.randperm(N, generator=gperm); pos = 0
    while step < total:
        model.train()
        if pos + bs > N:
            perm = torch.randperm(N, generator=gperm); pos = 0
        idx = perm[pos:pos + bs].to(DEVICE); pos += bs
        loss = crit(model(Xt[idx]), yt[idx])
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); step += 1
        if step % eval_every == 0 or step == total:
            f1 = metrics(yva, predict_head(model, Xva))['f1']
            if f1 > best + 1e-6:
                best, bad, best_state = f1, 0, copy.deepcopy(model.state_dict())
            else:
                bad += 1
                if bad >= CFG['head_patience']:
                    break
    model.load_state_dict(best_state); model.eval()
    return model, dict(steps=step, best_val_f1=best)

@torch.no_grad()
def predict_head(model, X, bs=4096, depol=0.0, shots=None, seed=0):
    model.eval()
    if depol > 0:
        bs = 256
    gen = torch.Generator(device=DEVICE).manual_seed(seed) if shots else None
    out = []
    for i in range(0, len(X), bs):
        xb = torch.as_tensor(X[i:i + bs], dtype=torch.float32, device=DEVICE)
        out.append(torch.softmax(model(xb, depol=depol, shots=shots, gen=gen), 1).cpu())
    return torch.cat(out).numpy()

def fit_logreg(Xtr, ytr, Xva, yva):
    best = None
    for C in [0.01, 0.1, 1.0]:
        m = LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(Xtr, ytr)
        f1 = metrics(yva, m.predict_proba(Xva))['f1']
        if best is None or f1 > best[0]:
            best = (f1, m, C)
    return best[1], dict(C=best[2], best_val_f1=best[0])
print('ok')

In [ ]:
# Cell 11: 5-fold patient-grouped CV x seeds on the head pool (model selection uses fold-val only)
Xh, yh, gh = FEAT['head']['X'], FEAT['head']['y'], FEAT['head']['g']
sgkf = StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=GLOBAL_SEED)
FOLDS = list(sgkf.split(Xh, yh, gh))
MODELS, SCALERS, CV_ROWS = {}, {}, []

for fold, (tri, vai) in enumerate(FOLDS):
    assert not set(gh[tri]) & set(gh[vai])
    sc = StandardScaler().fit(Xh[tri]); SCALERS[fold] = sc
    Xtr, Xva = sc.transform(Xh[tri]), sc.transform(Xh[vai]); ytr, yva = yh[tri], yh[vai]
    t0 = time.time(); m, info = fit_logreg(Xtr, ytr, Xva, yva); MODELS[('logreg', fold, 0)] = m
    for s in CFG['head_seeds']:      # logreg is deterministic: same score for every seed slot
        CV_ROWS.append(dict(fold=fold, seed=s, head='logreg', n_train=len(tri), n_val=len(vai),
                            secs=time.time() - t0, **metrics(yva, m.predict_proba(Xva))))
    for s in CFG['head_seeds']:
        for kd in TORCH_HEADS:
            t0 = time.time()
            m, info = train_head(kd, Xtr, ytr, Xva, yva, seed=1000 * fold + s)
            MODELS[(kd, fold, s)] = m
            CV_ROWS.append(dict(fold=fold, seed=s, head=kd, n_train=len(tri), n_val=len(vai),
                                secs=time.time() - t0, steps=info['steps'], **metrics(yva, predict_head(m, Xva))))
    cv_now = pd.DataFrame(CV_ROWS); cv_now = cv_now[cv_now.fold == fold].groupby('head')[['acc', 'f1']].mean()
    print(f'fold {fold}: ' + ' | '.join(f"{h}: f1 {cv_now.loc[h,'f1']:.4f}" for h in ALL_HEADS))

CV = pd.DataFrame(CV_ROWS); CV.to_csv(f'{OUT}/cv_runs.csv', index=False)

In [ ]:
# Cell 12: CV summary + Nadeau-Bengio corrected resampled t-test (hybrid_vqc vs each control)
def corrected_ttest(d, n_train, n_test):
    d = np.asarray(d, float); J = len(d); v = d.var(ddof=1)
    se = math.sqrt((1 / J + n_test / n_train) * v) if v > 0 else 0.0
    t = d.mean() / se if se > 0 else 0.0
    return t, (2 * stats.t.sf(abs(t), J - 1) if se > 0 else 1.0)

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

tab = CV.groupby('head')[['acc', 'bacc', 'f1']].agg(['mean', 'std']).reindex(ALL_HEADS)
print('CV (fold-val), mean +- std over folds x seeds:'); print((tab * 100).round(2).to_string())

ref = CV[CV['head'] == 'hybrid_vqc'].sort_values(['fold', 'seed']).set_index(['fold', 'seed'])
n_tr, n_te = CV.n_train.mean(), CV.n_val.mean()
rows = []
for h in [x for x in ALL_HEADS if x != 'hybrid_vqc']:
    o = CV[CV['head'] == h].sort_values(['fold', 'seed']).set_index(['fold', 'seed'])
    d = (ref.f1 - o.f1).values
    t, p = corrected_ttest(d, n_tr, n_te)
    rows.append(dict(comparison=f'hybrid_vqc - {h}', delta_f1_pp=100 * d.mean(), sd_pp=100 * d.std(ddof=1), t=t, p=p))
CV_TEST = pd.DataFrame(rows); CV_TEST['p_holm'] = holm(CV_TEST.p)
print('\nPaired comparisons on CV macro-F1 (corrected resampled t-test, Holm):')
print(CV_TEST.round(4).to_string(index=False)); CV_TEST.to_csv(f'{OUT}/cv_tests.csv', index=False)

plt.figure(figsize=(9, 4.5))
sns.boxplot(data=CV, x='head', y='f1', order=ALL_HEADS, color='#cfd8e3')
sns.stripplot(data=CV, x='head', y='f1', order=ALL_HEADS, color='#1f3b5c', size=4)
plt.xticks(range(len(ALL_HEADS)), [LABEL[h] for h in ALL_HEADS], rotation=15)
plt.ylabel('fold-val macro-F1'); plt.xlabel('')
plt.title(f"Patient-grouped {CFG['n_folds']}-fold CV x {len(CFG['head_seeds'])} seeds (identical frozen features)")
plt.tight_layout(); plt.savefig(f'{OUT}/cv_boxplot.png', dpi=150); plt.show()

In [ ]:
# Cell 13: data-efficiency (CV, fold-val only): where a parameter-efficiency claim would show up
DE_ROWS = []
DE_HEADS = ['logreg', 'classical_matched', 'hybrid_vqc', 'bottleneck_only']
for fold, (tri, vai) in enumerate(FOLDS):
    sc = SCALERS[fold]; Xva, yva = sc.transform(Xh[vai]), yh[vai]
    for npc in CFG['data_eff_per_class']:
        for s in CFG['head_seeds']:
            r = np.random.RandomState(7919 * fold + 31 * npc + s)
            sub = np.concatenate([r.choice(tri[yh[tri] == c], min(npc, (yh[tri] == c).sum()), replace=False)
                                  for c in range(K)])
            Xs, ys = sc.transform(Xh[sub]), yh[sub]
            for kd in DE_HEADS:
                if kd == 'logreg':
                    m, _ = fit_logreg(Xs, ys, Xva, yva); pr = m.predict_proba(Xva)
                else:
                    m, _ = train_head(kd, Xs, ys, Xva, yva, seed=s, min_steps=CFG['data_eff_min_steps'])
                    pr = predict_head(m, Xva)
                DE_ROWS.append(dict(fold=fold, seed=s, n_per_class=npc, head=kd, **metrics(yva, pr)))
    print(f'data-efficiency fold {fold} done')
DE = pd.DataFrame(DE_ROWS); DE.to_csv(f'{OUT}/data_efficiency.csv', index=False)
de = DE.groupby(['head', 'n_per_class']).f1.agg(['mean', 'std']).reset_index()
plt.figure(figsize=(7.5, 4.5))
for h in DE_HEADS:
    d = de[de['head'] == h]
    plt.errorbar(d.n_per_class, 100 * d['mean'], yerr=100 * d['std'], marker='o', capsize=3, label=LABEL[h])
plt.xscale('log'); plt.xlabel('training images per class'); plt.ylabel('fold-val macro-F1 (%)')
plt.title('Data efficiency (CV, mean +- std)'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/data_efficiency.png', dpi=150); plt.show()
print((de.pivot(index='n_per_class', columns='head', values='mean') * 100).round(2).to_string())

## Final evaluation — locked test & official test

Everything below this line is run **once**, after all settings are frozen. Each head's test prediction is the probability average of its 5 folds × seeds models (logreg: 5 fold models). Uncertainty: 95 % **patient-cluster bootstrap** CIs. Paired tests vs `hybrid_vqc`: exact McNemar and patient-cluster bootstrap of Δ macro-F1, Holm-corrected.

In [ ]:
# Cell 14: LOCKED TEST + OFFICIAL TEST (run once)
assert UNLOCK_TEST, 'Test sets are locked. Set UNLOCK_TEST=True only for the final, frozen run.'
open(f'{OUT}/TEST_UNLOCKED.log', 'a').write(f'{time.ctime()} locked_hash={locked_hash}\n')

def head_probs(h, X, **qkw):
    ps = []
    for (hh, fold, s), m in MODELS.items():
        if hh != h:
            continue
        Xs = SCALERS[fold].transform(X)
        ps.append(m.predict_proba(Xs) if h == 'logreg' else predict_head(m, Xs, **qkw))
    return np.mean(ps, 0)

def run_probs_per_model(h, X):
    out = []
    for (hh, fold, s), m in MODELS.items():
        if hh == h:
            Xs = SCALERS[fold].transform(X)
            out.append(m.predict_proba(Xs) if h == 'logreg' else predict_head(m, Xs))
    return out

def cluster_boot_idx(groups, B, seed):
    r = np.random.RandomState(seed); ug, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == k)[0] for k in range(len(ug))]
    for _ in range(B):
        yield np.concatenate([members[k] for k in r.randint(0, len(ug), len(ug))])

def ci(vals):
    return np.percentile(vals, [2.5, 97.5])

def clustered_mcnemar(correct_a, correct_b, groups):
    # McNemar for clustered matched pairs (Durkalski et al., Stat. Med. 2003): several images per patient
    a_only, b_only = (correct_a & ~correct_b).astype(int), (~correct_a & correct_b).astype(int)
    d = pd.DataFrame({'g': groups, 'a': a_only, 'b': b_only}).groupby('g').sum()
    diff = (d.a - d.b).values.astype(float)
    if (diff ** 2).sum() == 0:
        return 1.0
    return float(stats.chi2.sf(diff.sum() ** 2 / (diff ** 2).sum(), 1))

def mcnemar(y, pa, pb):
    a, b = pa.argmax(1) == y, pb.argmax(1) == y
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    p = stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 > 0 else 1.0
    return n01, n10, p

FINAL, TESTS, PROBS = [], [], {}
for split in ['locked_test', 'official_test']:
    X, y, g = FEAT[split]['X'], FEAT[split]['y'], FEAT[split]['g']
    P = {'backbone_e2e': FEAT[split]['bb_prob']}
    for h in ALL_HEADS:
        P[h] = head_probs(h, X)
    PROBS[split] = P
    boots = list(cluster_boot_idx(g, CFG['n_boot'], seed=123))
    for h, p in P.items():
        m = metrics(y, p)
        bf1 = [f1_score(y[i], p[i].argmax(1), average='macro') for i in boots]
        bacc = [accuracy_score(y[i], p[i].argmax(1)) for i in boots]
        per_run = [metrics(y, q)['f1'] for q in run_probs_per_model(h, X)] if h != 'backbone_e2e' else [m['f1']]
        FINAL.append(dict(split=split, head=h, n=len(y), patients=len(np.unique(g)),
                          acc=m['acc'], acc_lo=ci(bacc)[0], acc_hi=ci(bacc)[1], bacc=m['bacc'],
                          f1=m['f1'], f1_lo=ci(bf1)[0], f1_hi=ci(bf1)[1],
                          f1_runs_mean=np.mean(per_run), f1_runs_std=np.std(per_run, ddof=1) if len(per_run) > 1 else 0.0))
    for h in [x for x in ['backbone_e2e'] + ALL_HEADS if x != 'hybrid_vqc']:
        n01, n10, p_mc = mcnemar(y, P['hybrid_vqc'], P[h])
        d = [f1_score(y[i], P['hybrid_vqc'][i].argmax(1), average='macro') -
             f1_score(y[i], P[h][i].argmax(1), average='macro') for i in boots]
        dobs = metrics(y, P['hybrid_vqc'])['f1'] - metrics(y, P[h])['f1']
        p_boot = min(1.0, 2 * min(np.mean(np.array(d) <= 0), np.mean(np.array(d) >= 0)))
        TESTS.append(dict(split=split, comparison=f'hybrid_vqc vs {h}', delta_f1_pp=100 * dobs,
                          ci_lo_pp=100 * ci(d)[0], ci_hi_pp=100 * ci(d)[1],
                          vqc_only_correct=n01, other_only_correct=n10, p_mcnemar=p_mc, p_boot=p_boot,
                          p_mcnemar_clustered=clustered_mcnemar(P['hybrid_vqc'].argmax(1) == y, P[h].argmax(1) == y, g)))

FINAL = pd.DataFrame(FINAL); TESTS = pd.DataFrame(TESTS)
for split in TESTS.split.unique():
    msk = TESTS.split == split
    TESTS.loc[msk, 'p_mcnemar_holm'] = holm(TESTS.loc[msk, 'p_mcnemar'])
    TESTS.loc[msk, 'p_boot_holm'] = holm(TESTS.loc[msk, 'p_boot'])
    TESTS.loc[msk, 'p_mcnemar_clustered_holm'] = holm(TESTS.loc[msk, 'p_mcnemar_clustered'])
FINAL.to_csv(f'{OUT}/final_test_metrics.csv', index=False); TESTS.to_csv(f'{OUT}/final_test_comparisons.csv', index=False)

for split in ['locked_test', 'official_test']:
    f = FINAL[FINAL.split == split]
    print(f"\n=== {split}: {f.n.iloc[0]} images / {f.patients.iloc[0]} patients ===")
    for _, r in f.iterrows():
        print(f"{LABEL[r['head']]:28s} acc {100*r.acc:6.2f} [{100*r.acc_lo:.2f}, {100*r.acc_hi:.2f}]  "
              f"bacc {100*r.bacc:6.2f}  macro-F1 {100*r.f1:6.2f} [{100*r.f1_lo:.2f}, {100*r.f1_hi:.2f}]  "
              f"(single models {100*r.f1_runs_mean:.2f} +- {100*r.f1_runs_std:.2f})")
    print(TESTS[TESTS.split == split].drop(columns='split').round(4).to_string(index=False))

In [ ]:
# Cell 15: confusion matrices + per-class report for the trained hybrid VQC
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, split in zip(ax, ['locked_test', 'official_test']):
    y = FEAT[split]['y']; pred = PROBS[split]['hybrid_vqc'].argmax(1)
    cm = confusion_matrix(y, pred, labels=range(K))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=a)
    a.set_xlabel('predicted'); a.set_ylabel('true')
    a.set_title(f"{split}: hybrid VQC ({QC} x {QN}-qubit circuits, L={QL})")
    print(f'--- {split} ---'); print(classification_report(y, pred, target_names=CLASSES, digits=4))
plt.tight_layout(); plt.savefig(f'{OUT}/confusion_matrices.png', dpi=150); plt.show()

In [ ]:
# Cell 16: image-noise robustness in pixel space (stratified locked-test subset; backbone never saw noise)
NR = []
for kind in ['gaussian', 'speckle']:
    for lvl in CFG['img_noise_levels']:
        d = NOISE_FEAT[(kind, lvl)]
        NR.append(dict(kind=kind, level=lvl, head='backbone_e2e', **metrics(NOISE_Y, d['bb_prob'])))
        for h in ALL_HEADS:
            NR.append(dict(kind=kind, level=lvl, head=h, **metrics(NOISE_Y, head_probs(h, d['X']))))
NR = pd.DataFrame(NR); NR.to_csv(f'{OUT}/image_noise_robustness.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for a, kind in zip(ax, ['gaussian', 'speckle']):
    for h in ['backbone_e2e'] + ALL_HEADS:
        d = NR[(NR.kind == kind) & (NR['head'] == h)]
        a.plot(d.level * 100, d.f1 * 100, marker='o', label=LABEL[h])
    a.set_title(f'{kind} noise (pixel space, sigma as % of [0,1] range)')
    a.set_xlabel('noise sigma (%)'); a.grid(alpha=.3)
ax[0].set_ylabel('macro-F1 (%)'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/image_noise_robustness.png', dpi=150); plt.show()
print((NR.pivot_table(index=['kind', 'level'], columns='head', values='f1') * 100).round(2).to_string())

In [ ]:
# Cell 17: quantum noise -- finite shots and depolarising channels (trained hybrid VQC, locked-test subset)
Xn = NOISE_FEAT[('clean', 0.0)]['X']
classical_ref = metrics(NOISE_Y, head_probs('classical_matched', Xn))['f1']
QN_ROWS = []
for p in CFG['depol_levels']:
    QN_ROWS.append(dict(noise='depolarising', level=p, f1=metrics(NOISE_Y, head_probs('hybrid_vqc', Xn, depol=p))['f1']))
    print(f'depolarising p={p}: done')
for S_ in CFG['shot_levels']:
    vals = [metrics(NOISE_Y, head_probs('hybrid_vqc', Xn, shots=S_, seed=r))['f1'] for r in range(CFG['shot_repeats'])]
    QN_ROWS.append(dict(noise='shots', level=S_, f1=np.mean(vals), f1_std=np.std(vals)))
QNR = pd.DataFrame(QN_ROWS); QNR.to_csv(f'{OUT}/quantum_noise.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
d = QNR[QNR.noise == 'depolarising']
ax[0].plot(d.level, d.f1 * 100, 'o-', label='hybrid VQC'); ax[0].set_xscale('symlog', linthresh=1e-3)
ax[0].set_xlabel('depolarising probability per gate'); ax[0].set_title('Gate noise')
d = QNR[QNR.noise == 'shots']
ax[1].errorbar(d.level, d.f1 * 100, yerr=d.f1_std * 100, marker='o', capsize=3, label='hybrid VQC')
ax[1].set_xscale('log'); ax[1].invert_xaxis(); ax[1].set_xlabel('shots per expectation value')
ax[1].set_title('Shot noise')
for a in ax:
    a.axhline(classical_ref * 100, color='gray', ls='--', label='classical matched (noise-free)')
    a.set_ylabel('macro-F1 (%)'); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/quantum_noise.png', dpi=150); plt.show()
print(QNR.round(4).to_string(index=False))

In [ ]:
# Cell 18: circuit diagram (one of the parallel sub-circuits)
@qml.qnode(qml.device('default.qubit', wires=QN))
def draw_circ(x, w):
    qml.AngleEmbedding(x, wires=range(QN), rotation='Y')
    qml.StronglyEntanglingLayers(w, wires=range(QN))
    return [qml.expval(qml.PauliY(i)) for i in range(QN)]
try:
    fig, _ = qml.draw_mpl(draw_circ, level='device')(np.zeros(QN), np.zeros((QL, QN, 3)))
except TypeError:   # older PennyLane
    fig, _ = qml.draw_mpl(draw_circ, expansion_strategy='device')(np.zeros(QN), np.zeros((QL, QN, 3)))
fig.set_size_inches(14, 4)
fig.suptitle(f'One of {QC} parallel {QN}-qubit sub-circuits: RY embedding -> {QL} x StronglyEntanglingLayers -> <Y>', fontsize=10)
plt.savefig(f'{OUT}/circuit.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Cell 19: paper-ready summary
def fmt(r):
    return f"{100*r.f1:.2f} [{100*r.f1_lo:.2f}-{100*r.f1_hi:.2f}]"
cvm = CV.groupby('head').f1.agg(['mean', 'std'])
lines = ['| Model | Trainable middle params | CV macro-F1 (mean +- sd) | Locked test macro-F1 [95% CI] | Official test macro-F1 [95% CI] |',
         '|---|---|---|---|---|']
pm = dict(zip(PARAMS['head'], PARAMS.trainable_middle)); pm['logreg'] = '-'; pm['backbone_e2e'] = '-'
for h in ['backbone_e2e'] + ALL_HEADS:
    lt = FINAL[(FINAL.split == 'locked_test') & (FINAL['head'] == h)].iloc[0]
    ot = FINAL[(FINAL.split == 'official_test') & (FINAL['head'] == h)].iloc[0]
    cvs = f"{100*cvm.loc[h,'mean']:.2f} +- {100*cvm.loc[h,'std']:.2f}" if h in cvm.index else '-'
    lines.append(f"| {LABEL[h]} | {pm[h]} | {cvs} | {fmt(lt)} | {fmt(ot)} |")
table = '\n'.join(lines); print(table)
open(f'{OUT}/paper_table.md', 'w').write(table + '\n')

key = TESTS[(TESTS.split == 'locked_test') & (TESTS.comparison == 'hybrid_vqc vs classical_matched')].iloc[0]
cvk = CV_TEST[CV_TEST.comparison == 'hybrid_vqc - classical_matched'].iloc[0]
print(f"\nKey comparison, trained VQC vs parameter-matched classical block:\n"
      f"  CV:          delta macro-F1 = {cvk.delta_f1_pp:+.2f} pp (corrected t-test p = {cvk.p:.3g}, Holm {cvk.p_holm:.3g})\n"
      f"  locked test: delta macro-F1 = {key.delta_f1_pp:+.2f} pp, 95% CI [{key.ci_lo_pp:+.2f}, {key.ci_hi_pp:+.2f}], "
      f"McNemar p = {key.p_mcnemar:.3g} (Holm {key.p_mcnemar_holm:.3g})")
verdict = ('quantum layer outperforms the matched classical block' if key.ci_lo_pp > 0 and cvk.p_holm < 0.05 else
           'quantum layer underperforms the matched classical block' if key.ci_hi_pp < 0 and cvk.p_holm < 0.05 else
           'no detectable difference -> frame as parity at equal parameter count, not as a quantum advantage')
print('  verdict:', verdict)
json.dump(dict(locked_hash=locked_hash, verdict=verdict, cfg=CFG), open(f'{OUT}/summary.json', 'w'), indent=2)
print('\nAll outputs in', OUT)

In [ ]:
# Cell 20: record the software environment of this run (for the supplement / reviewers)
import platform, subprocess
env = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
with open(f'{OUT}/environment_01.txt', 'w') as f:
    f.write(f'python {platform.python_version()} | torch {torch.__version__} | cuda {torch.version.cuda} | device {gpu}\n\n' + env)
print('environment written to', f'{OUT}/environment_01.txt')